# Review 2 — Accelerating Deep Learning Operations
## Custom CUDA Kernels vs. PyTorch: ReLU and Matrix Multiplication

**Before anything else:** `Runtime -> Change runtime type -> T4 GPU` (or any CUDA GPU), then `Runtime -> Run all`.

This notebook is organized so each section maps directly onto a Review-2 checklist item:

1. Verify the CUDA environment
2. Identify the GPU
3. Verify PyTorch + CUDA
4. Write & compile a custom **ReLU** CUDA kernel
5. Run PyTorch's built-in ReLU
6. Correctness comparison — custom vs. PyTorch (ReLU)
7. Write & compile a custom tiled **matrix multiplication** CUDA kernel
8. Correctness comparison — custom vs. PyTorch (MatMul)
9. Benchmark both ops across sizes
10. Visualize results (the exact graphs to drop into your slides)

Every plot is saved into `results/` as a PNG so you can paste them straight into your presentation.


## Step 1 — Verify the CUDA environment

In [ ]:
!nvcc --version
!nvidia-smi

## Step 2 & 3 — Identify the GPU, verify PyTorch sees it

In [ ]:
import torch, time, os
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU runtime detected -> Runtime > Change runtime type > GPU"

device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"  Compute capability : {props.major}.{props.minor}")
print(f"  Total memory       : {props.total_memory / 1e9:.2f} GB")
print(f"  Multiprocessors    : {props.multi_processor_count}")

## Step 4 — Custom ReLU CUDA kernel

We compile the kernel on the fly with `torch.utils.cpp_extension.load_inline`, so there's no
separate `.cu` file or `setup.py` to manage — everything lives in this notebook.

`ReLU` is purely memory-bound (one read, one compare, one write per element), so a simple
grid-stride kernel is enough; there's little room to beat PyTorch here, and that's intentional —
it gives you a baseline "easy" case to contrast against matrix multiplication later.

In [ ]:
from torch.utils.cpp_extension import load_inline

relu_cuda_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void relu_kernel(const float* __restrict__ x, float* __restrict__ y, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) {
        float v = x[i];
        y[i] = v > 0.0f ? v : 0.0f;
    }
}

torch::Tensor relu_forward(torch::Tensor x) {
    TORCH_CHECK(x.is_cuda(), "input must be a CUDA tensor");
    TORCH_CHECK(x.scalar_type() == torch::kFloat32, "only float32 is supported");
    auto x_c = x.contiguous();
    auto y = torch::empty_like(x_c);
    int n = x_c.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    if (blocks > 65535) blocks = 65535;
    relu_kernel<<<blocks, threads>>>(x_c.data_ptr<float>(), y.data_ptr<float>(), n);
    return y;
}
'''

relu_cpp_source = "torch::Tensor relu_forward(torch::Tensor x);"

relu_ext = load_inline(
    name="custom_relu",
    cpp_sources=relu_cpp_source,
    cuda_sources=relu_cuda_source,
    functions=["relu_forward"],
    verbose=True,
)
print("Custom ReLU kernel compiled successfully.")

## Step 5 — PyTorch's built-in ReLU (baseline)

In [ ]:
x_demo = torch.randn(10, device=device)
print("input       :", x_demo)
print("torch.relu  :", torch.relu(x_demo))
print("custom relu :", relu_ext.relu_forward(x_demo))

## Step 6 — Correctness comparison: custom ReLU vs. PyTorch ReLU

In [ ]:
x = torch.randn(1_000_000, device=device)
y_custom = relu_ext.relu_forward(x)
y_torch  = torch.relu(x)

max_err = (y_custom - y_torch).abs().max().item()
print(f"Max absolute error: {max_err:.2e}")

assert torch.allclose(y_custom, y_torch, atol=1e-6), "Mismatch between custom and PyTorch ReLU!"
print("PASS: custom ReLU output is numerically identical to torch.relu")

## Step 7 — Custom matrix multiplication CUDA kernel

`C = A @ B`, with `A: [M,K]`, `B: [K,N]`, `C: [M,N]`.

This uses **shared-memory tiling**: each thread block loads a `TILE x TILE` tile of `A` and `B`
into fast on-chip shared memory and reuses every value `TILE` times before moving to the next
tile along `K`. This is the standard first optimization beyond a naive one-thread-per-output
kernel, and it's exactly the kind of kernel PyTorch's backend (cuBLAS) replaces with a far more
aggressively optimized implementation (register blocking, double buffering, vectorized loads,
Tensor Cores on supported GPUs) — which is the comparison this project is about.

In [ ]:
matmul_cuda_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define TILE 16

__global__ void matmul_kernel(const float* __restrict__ A, const float* __restrict__ B,
                               float* __restrict__ C, int M, int K, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;
    int col = blockIdx.x * TILE + tx;

    float acc = 0.0f;
    int numTiles = (K + TILE - 1) / TILE;

    for (int t = 0; t < numTiles; ++t) {
        int aCol = t * TILE + tx;
        As[ty][tx] = (row < M && aCol < K) ? A[row * K + aCol] : 0.0f;

        int bRow = t * TILE + ty;
        Bs[ty][tx] = (bRow < K && col < N) ? B[bRow * N + col] : 0.0f;

        __syncthreads();
#pragma unroll
        for (int i = 0; i < TILE; ++i) {
            acc += As[ty][i] * Bs[i][tx];
        }
        __syncthreads();
    }

    if (row < M && col < N) {
        C[row * N + col] = acc;
    }
}

torch::Tensor matmul_forward(torch::Tensor A, torch::Tensor B) {
    TORCH_CHECK(A.is_cuda() && B.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(A.scalar_type() == torch::kFloat32 && B.scalar_type() == torch::kFloat32,
                "only float32 is supported");
    auto A_c = A.contiguous();
    auto B_c = B.contiguous();

    int M = A_c.size(0), K = A_c.size(1), N = B_c.size(1);
    TORCH_CHECK(B_c.size(0) == K, "inner dimensions must match");

    auto C = torch::empty({M, N}, A_c.options());

    dim3 block(TILE, TILE);
    dim3 grid((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);
    matmul_kernel<<<grid, block>>>(A_c.data_ptr<float>(), B_c.data_ptr<float>(),
                                    C.data_ptr<float>(), M, K, N);
    return C;
}
'''

matmul_cpp_source = "torch::Tensor matmul_forward(torch::Tensor A, torch::Tensor B);"

matmul_ext = load_inline(
    name="custom_matmul",
    cpp_sources=matmul_cpp_source,
    cuda_sources=matmul_cuda_source,
    functions=["matmul_forward"],
    verbose=True,
)
print("Custom tiled MatMul kernel compiled successfully.")

## Step 8 — Correctness comparison: custom MatMul vs. PyTorch MatMul

In [ ]:
A = torch.randn(512, 512, device=device)
B = torch.randn(512, 512, device=device)

C_custom = matmul_ext.matmul_forward(A, B)
C_torch  = torch.matmul(A, B)

max_err = (C_custom - C_torch).abs().max().item()
rel_err = ((C_custom - C_torch).abs() / (C_torch.abs() + 1e-6)).max().item()
print(f"Max absolute error: {max_err:.4e}")
print(f"Max relative error: {rel_err:.4e}")

# Floating point accumulation order differs between kernels, so we use a
# slightly looser tolerance than the ReLU check (this is normal/expected).
assert torch.allclose(C_custom, C_torch, atol=1e-2, rtol=1e-3), "Mismatch between custom and PyTorch MatMul!"
print("PASS: custom tiled MatMul output matches torch.matmul within floating-point tolerance")

## Step 9 — Benchmark both ops across sizes

We use `torch.cuda.Event` for timing (accurate GPU-side timing, unlike `time.time()`), with a
warm-up phase so the first-call CUDA context/caching overhead doesn't pollute the measurement,
and we average over many iterations.

In [ ]:
def bench(fn, warmup=10, iters=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters  # ms per call


# --- ReLU across sizes ---
relu_sizes = [10_000, 100_000, 1_000_000, 10_000_000, 50_000_000]
relu_results = []
print(f"{'n':>12} | {'custom (ms)':>12} | {'torch (ms)':>11} | {'torch/custom':>12}")
for n in relu_sizes:
    x = torch.randn(n, device=device)
    t_custom = bench(lambda: relu_ext.relu_forward(x))
    t_torch  = bench(lambda: torch.relu(x))
    relu_results.append((n, t_custom, t_torch))
    print(f"{n:>12} | {t_custom:>12.4f} | {t_torch:>11.4f} | {t_torch / t_custom:>11.2f}x")

In [ ]:
# --- MatMul across sizes ---
matmul_sizes = [128, 256, 512, 1024, 2048]
matmul_results = []
print(f"{'N':>6} | {'custom (ms)':>12} | {'torch (ms)':>11} | {'custom GFLOP/s':>15} | {'torch GFLOP/s':>14} | {'torch/custom':>12}")
for s in matmul_sizes:
    A = torch.randn(s, s, device=device)
    B = torch.randn(s, s, device=device)
    t_custom = bench(lambda: matmul_ext.matmul_forward(A, B))
    t_torch  = bench(lambda: torch.matmul(A, B))
    flops = 2 * (s ** 3)
    g_custom = flops / (t_custom * 1e-3) / 1e9
    g_torch  = flops / (t_torch * 1e-3) / 1e9
    matmul_results.append((s, t_custom, t_torch, g_custom, g_torch))
    print(f"{s:>6} | {t_custom:>12.4f} | {t_torch:>11.4f} | {g_custom:>15.1f} | {g_torch:>14.1f} | {t_torch / t_custom:>11.2f}x")

## Step 10 — Visualize results (presentation-ready graphs)

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("results", exist_ok=True)

sizes    = [r[0] for r in relu_results]
custom_t = [r[1] for r in relu_results]
torch_t  = [r[2] for r in relu_results]

plt.figure(figsize=(7, 5))
plt.plot(sizes, custom_t, marker="o", label="Custom CUDA kernel")
plt.plot(sizes, torch_t, marker="s", label="PyTorch (torch.relu)")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Number of elements"); plt.ylabel("Latency (ms)")
plt.title("ReLU: Custom CUDA kernel vs. PyTorch")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("results/relu_benchmark.png", dpi=150)
plt.show()

In [ ]:
Ns          = [r[0] for r in matmul_results]
mm_custom_t = [r[1] for r in matmul_results]
mm_torch_t  = [r[2] for r in matmul_results]
mm_custom_g = [r[3] for r in matmul_results]
mm_torch_g  = [r[4] for r in matmul_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(Ns, mm_custom_t, marker="o", label="Custom CUDA kernel")
axes[0].plot(Ns, mm_torch_t, marker="s", label="PyTorch (torch.matmul / cuBLAS)")
axes[0].set_xlabel("Matrix size (N x N)"); axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("MatMul Latency"); axes[0].set_yscale("log")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(Ns, mm_custom_g, marker="o", label="Custom CUDA kernel")
axes[1].plot(Ns, mm_torch_g, marker="s", label="PyTorch (torch.matmul / cuBLAS)")
axes[1].set_xlabel("Matrix size (N x N)"); axes[1].set_ylabel("Throughput (GFLOP/s)")
axes[1].set_title("MatMul Throughput")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/matmul_benchmark.png", dpi=150)
plt.show()

In [ ]:
speedups = [t / c for c, t in zip(mm_custom_t, mm_torch_t)]

plt.figure(figsize=(7, 5))
plt.bar([str(n) for n in Ns], speedups, color="#4C72B0")
plt.axhline(1.0, color="red", linestyle="--", label="Parity (1.0x)")
plt.xlabel("Matrix size (N x N)"); plt.ylabel("PyTorch/cuBLAS speedup over custom kernel (x)")
plt.title("How much faster is PyTorch/cuBLAS than our naive tiled kernel?")
plt.legend()
plt.tight_layout()
plt.savefig("results/matmul_speedup.png", dpi=150)
plt.show()

print("Saved: results/relu_benchmark.png, results/matmul_benchmark.png, results/matmul_speedup.png")
print("Download these (folder icon on the left in Colab -> results/ -> right-click -> Download)")
print("and drop them into the matching slides of your presentation.")

## What to expect (for your discussion section)

- **ReLU** is memory-bound (one read + one write per element). Your custom kernel and PyTorch's
  built-in kernel usually land close to each other, both near the GPU's memory-bandwidth limit —
  this is the case where hand-written CUDA has the least room to add value.
- **MatMul** is compute-bound. `torch.matmul` calls into **cuBLAS**, which uses register
  blocking, double buffering, vectorized memory access, and (on supported GPUs) **Tensor Cores**.
  Our single-level shared-memory tile is the *first* optimization step, not the last — expect the
  gap between the two to **grow** as matrix size increases.
- Report both **latency (ms)** and **throughput (GFLOP/s)** — GFLOP/s is what lets you compare
  fairly across different matrix sizes.

## Next steps (Review 3 preview)

- Add more layers: Conv2d, Softmax, LayerNorm (reduction-based kernels)
- Deeper MatMul optimizations: register blocking, double buffering, `float4` vectorized loads
- Profile with NVIDIA Nsight Compute to see occupancy and memory-bandwidth utilization directly
- Compare against `torch.compile` / TorchScript-fused kernels, not just eager-mode PyTorch
